# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to efficiently load, explore, and analyze the FAIR<sup>2</sup> dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The data is described via the [Croissant schema](https://mlcommons.org/croissant/) at:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure required packages are installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records using the `mlcroissant` library. Both the overview (metadata) and the record data will be retrieved for investigation.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
List all available record sets (tables), their field `@id`s (columns), and a preview of the table to help select them for further extraction.


In [ ]:
# Get all record set @ids (unique identifiers)
record_sets = [r["@id"] for r in metadata._json.get("recordSet", [])]
if not record_sets:
    # Try metadata.recordSet which may be parsed as a list of Croissant objects in new mlcroissant
    try:
        record_sets = [r['@id'] for r in metadata.recordSet]
    except Exception:
        pass

print("Available record sets (@id):", record_sets)

# For demonstration, display each record set's fields and their @ids
for rsid in record_sets:
    print(f"\nRecordSet @id: {rsid}")
    # Find the actual RecordSet dict
    recset = next((r for r in metadata._json.get('recordSet', []) if r['@id']==rsid), None)
    if recset is None:
        continue
    # Each record set lists 'field', each as a dict or @id
    fields = recset.get('field', [])
    # Normalize field dicts (dereference if '@id' only)
    fields_descriptions = []
    for f in fields:
        if isinstance(f, dict):
            fid = f.get('@id', f)
            label = f.get('name', fid)
        else:
            fid = f
            label = f
        fields_descriptions.append((fid, label))
    print("  Fields (@id, label):")
    for fid, label in fields_descriptions:
        print(f"    - {fid:40s} | {label}")

## 3. Data Extraction
Let's extract data from **all record sets** into pandas DataFrames for further exploration. Use the record set and field `@id`s identified above.


In [ ]:
# Load all data into DataFrames for easy manipulation
dataframes = {}
for rsid in record_sets:
    print(f"Loading records for {rsid} ...")
    try:
        records = list(dataset.records(record_set=rsid))  # All records as list of dicts
        dataframes[rsid] = pd.DataFrame(records)
        print(f"  Columns: {list(dataframes[rsid].columns)}  | Rows: {len(dataframes[rsid])}")
    except Exception as e:
        print(f"  Failed: {e}")

if dataframes:
    # Pick the first available record set as main example in further steps
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nMain record set for EDA: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No record sets could be loaded for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply basic analysis: filtering, normalization, grouping.

**Note:** Use examples from the actual data above. For biology records, you might filter by the `coverage` or `MW` (molecular weight) field when available. All fields referenced by their `@id`.

In [ ]:
# Pick a numeric field by @id from the main record set
# Example field @ids as seen in the previous output (replace with actual ids for your data)
# For demonstration, we'll try common protein table ids
possible_numeric_ids = ['coverage', 'cr:coverage', 'MolWt', 'cr:MolWt', 'MW', 'cr:MW', 'peptide_count', 'cr:peptide_count',
                        'cr:unique_peptides', 'cr:abundance', 'cr:pI']

df = dataframes.get(main_record_set_id)
print("Available columns:", df.columns.tolist())
numeric_field_id = None
for candidate in possible_numeric_ids:
    if candidate in df.columns:
        numeric_field_id = candidate
        break

if numeric_field_id is None:
    # Fallback: pick first float/integer column
    for c in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                break
        except Exception:
            continue

print(f"Using numeric field for analysis: {numeric_field_id}")

# Filtering: keep values > lower threshold
if numeric_field_id is not None:
    # Ensure it's float
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a categorical field if available
    # Try common group fields: 'accession', 'cr:accession', 'description', 'cr:description', etc.
    possible_group_fields = ['accession', 'cr:accession', 'description', 'cr:description', 'sample', 'cr:sample', 'group', 'cr:group']
    group_field = None
    for candidate in possible_group_fields:
        if candidate in df.columns:
            group_field = candidate
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field} (showing up to 5 groups):")
        print(grouped_df.head())
    else:
        print("\nNo suitable group field found.")
else:
    print("No numeric field suitable for analysis in main record set.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and the normalized values if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and df[numeric_field_id].notnull().sum() > 0:
    plt.figure(figsize=(10,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True)
        plt.title(f"Normalized {numeric_field_id} distribution (filtered)")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel("Count")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, you learned how to load the FAIR<sup>2</sup> protein mass spectrometry dataset using the `mlcroissant` library. You explored the dataset structure using Croissant schema `@id`s for each major data entity, loaded the record sets into DataFrames, and performed a first round of simple exploratory data analysis and visualization.

For in-depth analysis, you are encouraged to:
- Dive into documentation files or extra data fields for protein annotation
- Use additional grouping and filters specific to your biological hypotheses
- Save the processed DataFrame for downstream machine learning or statistical modeling!

**Reference:** [FAIR² mass spectrometry - SenScience (2026)](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
